In [1]:
import pandas as pd
import requests
import json
import time
from pathlib import Path
from datetime import datetime, timezone

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELING_DIR = PROJECT_ROOT / "data" / "modeling"

print("Project root:", PROJECT_ROOT)
print("Pandas version:", pd.__version__)

Project root: /Users/xinyishen/Desktop/Brown/DATA 1030/DATA1030-fall2026/DATA1030-Midterm-Xinfinity
Pandas version: 3.0.3


In [3]:
BASE_URL = "https://api.nhtsa.gov/complaints/complaintsByVehicle"

In [4]:
#test API connection
params = {
    "make": "Toyota",
    "model": "Camry",
    "modelYear": 2020
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=30
)

response.raise_for_status()

data = response.json()

print("Status Code:", response.status_code)
print("Response Keys:", list(data.keys()))
print("Number of complaints:", len(data.get("results", [])))

Status Code: 200
Response Keys: ['count', 'message', 'results']
Number of complaints: 274


In [5]:
complaints = data["results"]

df_raw = pd.DataFrame(complaints)

print("Number of rows:", df_raw.shape[0])
print("Number of columns:", df_raw.shape[1])

print("\nAvailable columns:")
print(df_raw.columns.tolist())

display(df_raw.head(3))

Number of rows: 274
Number of columns: 12

Available columns:
['odiNumber', 'manufacturer', 'crash', 'fire', 'numberOfInjuries', 'numberOfDeaths', 'dateOfIncident', 'dateComplaintFiled', 'vin', 'components', 'summary', 'products']


,odiNumber,manufacturer,crash,fire,numberOfInjuries,numberOfDeaths,dateOfIncident,dateComplaintFiled,vin,components,summary,products
0,11766564,Toyota Motor Corporation,False,False,0,0,09/05/2026,09/23/2026,4T1C11AK1LU,FUEL/PROPULSION SYSTEM,"While fixing another problem, I pulled the cod...","[{'type': 'Vehicle', 'productYear': '2020', 'p..."
1,11765004,Toyota Motor Corporation,False,False,0,0,06/01/2021,09/17/2026,4T1K61AK7LU,"ELECTRICAL SYSTEM,UNKNOWN OR OTHER,FUEL/PROPUL...",My car sputters and dies while driving. It loo...,"[{'type': 'Vehicle', 'productYear': '2020', 'p..."
2,11764646,Toyota Motor Corporation,False,False,0,0,08/10/2026,09/16/2026,4t1g11ak0lu,POWER TRAIN,The transmission is premature wearing due to e...,"[{'type': 'Vehicle', 'productYear': '2020', 'p..."


In [6]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 274 entries, 0 to 273
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   odiNumber           274 non-null    int64 
 1   manufacturer        274 non-null    str   
 2   crash               274 non-null    bool  
 3   fire                274 non-null    bool  
 4   numberOfInjuries    274 non-null    int64 
 5   numberOfDeaths      274 non-null    int64 
 6   dateOfIncident      274 non-null    str   
 7   dateComplaintFiled  274 non-null    str   
 8   vin                 274 non-null    str   
 9   components          274 non-null    str   
 10  summary             274 non-null    str   
 11  products            274 non-null    object
dtypes: bool(2), int64(3), object(1), str(6)
memory usage: 22.1+ KB


In [7]:
# Missing value summary
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (
        df_raw.isna().mean() * 100
    ).round(2)
}).sort_values("missing_pct", ascending=False)

display(missing_summary)

,missing_count,missing_pct
odiNumber,0,0.0
manufacturer,0,0.0
crash,0,0.0
fire,0,0.0
numberOfInjuries,0,0.0
numberOfDeaths,0,0.0
dateOfIncident,0,0.0
dateComplaintFiled,0,0.0
vin,0,0.0
components,0,0.0


In [8]:
print("API reported count:", data.get("count"))
print("Records retrieved:", len(df_raw))

assert data.get("count") == len(df_raw), (
    "API count does not match retrieved records"
)

print("PASS: API count matches retrieved records")

API reported count: 274
Records retrieved: 274
PASS: API count matches retrieved records


In [9]:
for col in df_raw.columns:
    print(f"\n--- {col} ---")
    print("Unique values:", df_raw[col].astype(str).nunique())
    print(df_raw[col].astype(str).value_counts().head(15))


--- odiNumber ---
Unique values: 274
odiNumber
11766564    1
11765004    1
11764646    1
11762945    1
11760803    1
11760393    1
11756265    1
11754649    1
11754116    1
11753736    1
11752925    1
11747189    1
11746558    1
11745192    1
11740255    1
Name: count, dtype: int64

--- manufacturer ---
Unique values: 1
manufacturer
Toyota Motor Corporation    274
Name: count, dtype: int64

--- crash ---
Unique values: 2
crash
False    260
True      14
Name: count, dtype: int64

--- fire ---
Unique values: 2
fire
False    272
True       2
Name: count, dtype: int64

--- numberOfInjuries ---
Unique values: 4
numberOfInjuries
0    262
2      6
1      4
3      2
Name: count, dtype: int64

--- numberOfDeaths ---
Unique values: 1
numberOfDeaths
0    274
Name: count, dtype: int64

--- dateOfIncident ---
Unique values: 253
dateOfIncident
12/20/2023    5
08/01/2020    3
09/02/2020    3
03/11/2026    2
12/31/1969    2
01/05/2026    2
12/15/2025    2
12/04/2025    2
11/11/2025    2
01/13/2025   

## Choose variables

In [10]:
vehicles = {
    "Toyota": ["Camry"],
    "Honda": ["Civic"],
    "Ford": ["Escape", "F-150"],
    "Hyundai": ["Elantra"],
    "Nissan": ["Altima"],
    "BMW": ["3 Series"],
    "Mercedes-Benz": ["C-Class"],
    "Audi": ["Q5"],
    "Volkswagen": ["Tiguan"]
}
years = [2018, 2019, 2020]

print("Planned requests:", sum(len(models) for models in vehicles.values()) * len(years))

Planned requests: 30


## Get complaints from the API

In [11]:
all_records = []
request_log = []

for make, models in vehicles.items():
    for model in models:
        for year in years:
            params = {"make": make, "model": model, "modelYear": year}
            try:
                response = requests.get(BASE_URL, params=params, timeout=60)
                response.raise_for_status()
                result = response.json()
                records = result["results"]
                if result["count"] != len(records):
                    raise ValueError("API count does not match")

                for record in records:
                    row = record.copy()
                    row["query_make"] = make
                    row["query_model"] = model
                    row["query_model_year"] = year
                    all_records.append(row)

                status = "success"
                print(make, model, year, len(records))
            except (requests.RequestException, ValueError, KeyError, TypeError) as e:
                status = "failed"
                records = []
                print("FAILED:", make, model, year, e)

            request_log.append({"make": make, "model": model,
                                "year": year, "count": len(records),
                                "status": status})
            time.sleep(0.5)

df_raw = pd.DataFrame(all_records)
df_request_log = pd.DataFrame(request_log)
display(df_request_log)

Toyota Camry 2018 747
Toyota Camry 2019 397
Toyota Camry 2020 274
Honda Civic 2018 596
Honda Civic 2019 346
Honda Civic 2020 227
Ford Escape 2018 1806
Ford Escape 2019 594
Ford Escape 2020 1533
Ford F-150 2018 83
Ford F-150 2019 56
Ford F-150 2020 48
Hyundai Elantra 2018 283
Hyundai Elantra 2019 251
Hyundai Elantra 2020 253
Nissan Altima 2018 226
Nissan Altima 2019 232
Nissan Altima 2020 179
BMW 3 Series 2018 59
BMW 3 Series 2019 41
BMW 3 Series 2020 53
Mercedes-Benz C-Class 2018 130
Mercedes-Benz C-Class 2019 79
Mercedes-Benz C-Class 2020 46
Audi Q5 2018 207
Audi Q5 2019 52
Audi Q5 2020 25
Volkswagen Tiguan 2018 284
Volkswagen Tiguan 2019 192
Volkswagen Tiguan 2020 141


,make,model,year,count,status
0,Toyota,Camry,2018,747,success
1,Toyota,Camry,2019,397,success
2,Toyota,Camry,2020,274,success
3,Honda,Civic,2018,596,success
4,Honda,Civic,2019,346,success
5,Honda,Civic,2020,227,success
6,Ford,Escape,2018,1806,success
7,Ford,Escape,2019,594,success
8,Ford,Escape,2020,1533,success
9,Ford,F-150,2018,83,success


In [12]:
print("Requests:", len(df_request_log))
print("Failed requests:", (df_request_log["status"] == "failed").sum())
print("Zero-result requests:", ((df_request_log["status"] == "success") & (df_request_log["count"] == 0)).sum())
display(df_request_log[df_request_log["status"] != "success"])
display(df_request_log[(df_request_log["status"] == "success") & (df_request_log["count"] == 0)])

print("Rows:", len(df_raw))
if not df_raw.empty:
    print("Unique complaint IDs:", df_raw["odiNumber"].nunique())
    print("Repeated IDs across queries:", df_raw["odiNumber"].duplicated().sum())

Requests: 30
Failed requests: 0
Zero-result requests: 0


,make,model,year,count,status


,make,model,year,count,status


Rows: 9440
Unique complaint IDs: 9440
Repeated IDs across queries: 0


In [13]:
display(df_raw.drop(columns=["vin", "summary", "products"], errors="ignore").head())
print(df_raw.shape)
df_raw.info()

,odiNumber,manufacturer,crash,fire,numberOfInjuries,numberOfDeaths,dateOfIncident,dateComplaintFiled,components,query_make,query_model,query_model_year
0,11769969,Toyota Motor Corporation,False,False,0,0,10/06/2026,10/08/2026,ENGINE,Toyota,Camry,2018
1,11769124,Toyota Motor Corporation,False,False,0,0,10/03/2026,10/05/2026,POWER TRAIN,Toyota,Camry,2018
2,11768868,Toyota Motor Corporation,False,False,0,0,10/02/2026,10/04/2026,FUEL/PROPULSION SYSTEM,Toyota,Camry,2018
3,11768335,Toyota Motor Corporation,False,False,0,0,09/29/2026,10/01/2026,POWER TRAIN,Toyota,Camry,2018
4,11767971,Toyota Motor Corporation,False,False,0,0,09/28/2026,09/30/2026,STEERING,Toyota,Camry,2018


(9440, 15)
<class 'pandas.DataFrame'>
RangeIndex: 9440 entries, 0 to 9439
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   odiNumber           9440 non-null   int64 
 1   manufacturer        9440 non-null   str   
 2   crash               9440 non-null   bool  
 3   fire                9440 non-null   bool  
 4   numberOfInjuries    9440 non-null   int64 
 5   numberOfDeaths      9440 non-null   int64 
 6   dateOfIncident      9440 non-null   str   
 7   dateComplaintFiled  9440 non-null   str   
 8   vin                 9417 non-null   str   
 9   components          9440 non-null   str   
 10  summary             9440 non-null   str   
 11  products            9440 non-null   object
 12  query_make          9440 non-null   str   
 13  query_model         9440 non-null   str   
 14  query_model_year    9440 non-null   int64 
dtypes: bool(2), int64(4), object(1), str(8)
memory usage: 977.3+ KB


## Check for missing values and distributions

In [14]:
print("Pandas missing values:")
display(df_raw.isna().sum())

print("Blank strings:")
text_cols = ["manufacturer", "vin", "components", "summary"]
for col in text_cols:
    if col in df_raw:
        print(col, df_raw[col].astype("string").str.strip().eq("").sum())

if "products" in df_raw:
    print("Empty product lists:", df_raw["products"].map(lambda x: isinstance(x, list) and len(x) == 0).sum())

Pandas missing values:


odiNumber              0
manufacturer           0
crash                  0
fire                   0
numberOfInjuries       0
numberOfDeaths         0
dateOfIncident         0
dateComplaintFiled     0
vin                   23
components             0
summary                0
products               0
query_make             0
query_model            0
query_model_year       0
dtype: int64

Blank strings:
manufacturer 0
vin 176
components 0
summary 0
Empty product lists: 0


In [15]:
for col in ["manufacturer", "crash", "fire", "numberOfInjuries", "numberOfDeaths", "components"]:
    print("\n", col)
    display(df_raw[col].value_counts(dropna=False).head(10))


 manufacturer


manufacturer
Ford Motor Company                    4120
Toyota Motor Corporation              1418
Honda (American Honda Motor Co.)      1169
Volkswagen Group of America, Inc.      901
Hyundai Motor America                  787
Nissan North America, Inc.             637
Mercedes-Benz USA, LLC                 254
BMW of North America, LLC              153
Continental Tire the Americas, LLC       1
Name: count, dtype: int64


 crash


crash
False    9072
True      368
Name: count, dtype: int64


 fire


...

numberOfInjuries
0     9197
1      182
2       49
4        5
3        4
10       1
6        1
5        1
Name: count, dtype: int64


 numberOfDeaths


numberOfDeaths
0    9435
1       4
2       1
Name: count, dtype: int64


 components


components
ENGINE                       1716
POWER TRAIN                   816
UNKNOWN OR OTHER              761
STRUCTURE                     641
STEERING                      424
ENGINE AND ENGINE COOLING     371
ELECTRICAL SYSTEM             350
FUEL SYSTEM, GASOLINE         296
FUEL/PROPULSION SYSTEM        269
SERVICE BRAKES                207
Name: count, dtype: int64

## Check the vehicle info in `products`

In [16]:
matched = []
for _, row in df_raw.iterrows():
    found = False
    if isinstance(row["products"], list):
        for p in row["products"]:
            if (isinstance(p, dict)
                and p.get("type") == "Vehicle"
                and str(p.get("productMake", "")).upper() == row["query_make"].upper()
                and str(p.get("productModel", "")).upper() == row["query_model"].upper()
                and str(p.get("productYear", "")) == str(row["query_model_year"])):
                found = True
    matched.append(found)

df_raw["vehicle_match"] = matched
print(df_raw["vehicle_match"].value_counts(dropna=False))
display(df_raw.loc[~df_raw["vehicle_match"],
                   ["odiNumber", "query_make", "query_model", "query_model_year"]].head(10))

vehicle_match
True     9032
False     408
Name: count, dtype: int64


,odiNumber,query_make,query_model,query_model_year
8131,11756464,BMW,3 Series,2018
8132,11678001,BMW,3 Series,2018
8133,11670096,BMW,3 Series,2018
8134,11667410,BMW,3 Series,2018
8135,11660219,BMW,3 Series,2018
8136,11655822,BMW,3 Series,2018
8137,11648186,BMW,3 Series,2018
8138,11646108,BMW,3 Series,2018
8139,11641715,BMW,3 Series,2018
8140,11637952,BMW,3 Series,2018


In [17]:
df_raw.groupby(
    ["query_make", "query_model"]
)["vehicle_match"].value_counts().unstack(fill_value=0)

vehicle_match              True   False
query_make    query_model              
Audi          Q5             284      0
BMW           3 Series         0    153
Ford          Escape        3933      0
              F-150          187      0
Honda         Civic         1169      0
Hyundai       Elantra        787      0
Mercedes-Benz C-Class          0    255
Nissan        Altima         637      0
Toyota        Camry         1418      0
Volkswagen    Tiguan         617      0

In [18]:
for model in ["3 Series", "C-Class"]:
    subset = df_raw[
        df_raw["query_model"] == model
    ]

    print(f"\n--- {model} ---")

    print("Example products:")
    display(subset["products"].head(3).tolist())


--- 3 Series ---
Example products:


[[{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'BMW',
   'productModel': '330E',
   'manufacturer': 'BMW of North America, LLC'}],
 [{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'BMW',
   'productModel': '330I',
   'manufacturer': 'BMW of North America, LLC'}],
 [{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'BMW',
   'productModel': '328D',
   'manufacturer': 'BMW of North America, LLC'}]]


--- C-Class ---
Example products:


[[{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'MERCEDES-BENZ',
   'productModel': 'C 300',
   'manufacturer': 'Mercedes-Benz USA, LLC'}],
 [{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'MERCEDES BENZ',
   'productModel': 'C 300 4MATIC',
   'manufacturer': 'Mercedes-Benz USA, LLC'}],
 [{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'MERCEDES-BENZ',
   'productModel': 'C 300',
   'manufacturer': 'Mercedes-Benz USA, LLC'}]]

## Identify falsely categorized

In [19]:
# Get vehicle model from products
def get_vehicle_model(products):
    for p in products:
        if p.get("type") == "Vehicle":
            return p.get("productModel")
    return None

df_raw["product_model"] = df_raw["products"].apply(get_vehicle_model)

In [20]:
display(
    df_raw.groupby(["query_make", "product_model"])
    .size()
    .reset_index(name="count")
)

,query_make,product_model,count
0,Audi,Q5,284
1,BMW,320I,16
2,BMW,328D,7
3,BMW,330E,6
4,BMW,330I,121
5,BMW,340I,3
6,Ford,ESCAPE,3933
7,Ford,F-150,187
8,Honda,CIVIC,1169
9,Hyundai,ELANTRA,787


In [21]:
model_mapping = {
    "320I": "3 SERIES",
    "328D": "3 SERIES",
    "330E": "3 SERIES",
    "330I": "3 SERIES",
    "340I": "3 SERIES",

    "AMG C43": "C-CLASS",
    "AMG C63": "C-CLASS",
    "AMG C63 S": "C-CLASS",
    "AMG C63S": "C-CLASS",
    "C 250 D": "C-CLASS",
    "C 300": "C-CLASS",
    "C 300 4MATIC": "C-CLASS",
    "C 350 E": "C-CLASS",
    "C300": "C-CLASS",
    "C350": "C-CLASS",
    "C205": "C-CLASS"
}

In [22]:
df_clean = df_raw.copy()

df_clean["product_model"] = (
    df_clean["product_model"].str.upper().str.strip()
)

df_clean["make"] = (
    df_clean["query_make"].str.upper().str.strip()
)

df_clean["model"] = (
    df_clean["product_model"]
    .replace(model_mapping)
)

df_clean["model_year"] = df_clean["query_model_year"]

In [23]:
display(
    df_clean.groupby(["make", "model"])
    .size()
    .reset_index(name="count")
)

,make,model,count
0,AUDI,Q5,284
1,BMW,3 SERIES,153
2,FORD,ESCAPE,3933
3,FORD,F-150,187
4,HONDA,CIVIC,1169
5,HYUNDAI,ELANTRA,787
6,MERCEDES-BENZ,C-CLASS,255
7,NISSAN,ALTIMA,637
8,TOYOTA,CAMRY,1418
9,VOLKSWAGEN,TIGUAN,617


In [24]:
df_raw.loc[
    df_raw["product_model"] == "PROCONTACT GX SSR MOE",
    ["odiNumber", "query_make", "query_model", "products"]
]

,odiNumber,query_make,query_model,products


In [25]:
# df_raw.loc[8386, "products"]

In [26]:
df_raw.loc[
    df_raw["product_model"] == "C205",
    ["odiNumber", "query_model_year", "products"]
]

,odiNumber,query_model_year,products
8435,11680791,2019,"[{'type': 'Vehicle', 'productYear': '2019', 'p..."


In [27]:
df_raw.loc[8435, "products"]

[{'type': 'Vehicle',
  'productYear': '2019',
  'productMake': 'MERCEDES BENZ',
  'productModel': 'C205',
  'manufacturer': 'Mercedes-Benz USA, LLC'}]

In [28]:
df_clean.groupby(["make", "model"]).size()

make           model   
AUDI           Q5           284
BMW            3 SERIES     153
FORD           ESCAPE      3933
               F-150        187
HONDA          CIVIC       1169
HYUNDAI        ELANTRA      787
MERCEDES-BENZ  C-CLASS      255
NISSAN         ALTIMA       637
TOYOTA         CAMRY       1418
VOLKSWAGEN     TIGUAN       617
dtype: int64

# Data Cleaning

## Convert dates & Find delays

In [29]:
df_clean["filed_date"] = pd.to_datetime(
    df_clean["dateComplaintFiled"],
    format="%m/%d/%Y",
    errors="coerce"
)

df_clean["incident_date"] = pd.to_datetime(
    df_clean["dateOfIncident"],
    format="%m/%d/%Y",
    errors="coerce"
)

In [30]:
print("Missing filed dates:", df_clean["filed_date"].isna().sum())
print("Missing incident dates:", df_clean["incident_date"].isna().sum())

print("Incident before 1980:",
      (df_clean["incident_date"].dt.year < 1980).sum())

print("Incident after filing:",
      (df_clean["incident_date"] > df_clean["filed_date"]).sum())

Missing filed dates: 0
Missing incident dates: 0
Incident before 1980: 18
Incident after filing: 0


In [31]:
df_clean["reporting_delay"] = (
    df_clean["filed_date"] - df_clean["incident_date"]
).dt.days

df_clean["reporting_delay"].describe()

count     9440.000000
mean       137.461123
std       1006.981852
min          0.000000
25%          2.000000
50%         13.000000
75%         61.000000
max      37015.000000
Name: reporting_delay, dtype: float64

In [32]:
df_clean.loc[
    df_clean["incident_date"].dt.year < 1980,
    ["odiNumber", "make", "model", "model_year",
     "incident_date", "filed_date", "reporting_delay"]
]

,odiNumber,make,model,model_year,incident_date,filed_date,reporting_delay
88,11702696,TOYOTA,CAMRY,2018,1969-12-31,2025-12-03,20426
1031,11387502,TOYOTA,CAMRY,2019,1919-09-09,2021-01-11,37015
1171,11718234,TOYOTA,CAMRY,2020,1969-12-31,2026-02-16,20501
1196,11690118,TOYOTA,CAMRY,2020,1969-12-31,2025-09-27,20359
2377,11736426,HONDA,CIVIC,2020,1969-12-31,2026-05-07,20581
2701,11746148,FORD,ESCAPE,2018,1969-12-31,2026-06-24,20629
2752,11737407,FORD,ESCAPE,2018,1969-12-31,2026-05-12,20586
2773,11733178,FORD,ESCAPE,2018,1969-12-31,2026-04-23,20567
2850,11714461,FORD,ESCAPE,2018,1969-12-31,2026-01-30,20484
4402,11764367,FORD,ESCAPE,2019,1969-12-31,2026-09-15,20712


In [34]:
df_clean["invalid_incident_date"] = (
    df_clean["incident_date"].dt.year < 1980
)

In [35]:
df_clean["reporting_delay_clean"] = (
    df_clean["reporting_delay"].where(
        ~df_clean["invalid_incident_date"]
    )
)

df_clean["reporting_delay_clean"].describe()

count     9422.000000
mean        96.745065
std        345.238995
min          0.000000
25%          2.000000
50%         12.000000
75%         61.000000
max      15778.000000
Name: reporting_delay_clean, dtype: float64

In [36]:
print("Total rows:", len(df_clean))
print("Unique ODI numbers:", df_clean["odiNumber"].nunique())
print("Duplicate ODI numbers:",
      df_clean["odiNumber"].duplicated().sum())

Total rows: 9440
Unique ODI numbers: 9440
Duplicate ODI numbers: 0


In [37]:
duplicates = df_clean[
    df_clean.duplicated("odiNumber", keep=False)
]

display(duplicates[
    ["odiNumber", "make", "model",
     "model_year", "components"]
].head(20))

,odiNumber,make,model,model_year,components


## Missing value handling

In [38]:
df_clean["vin"] = df_clean["vin"].replace(
    r"^\s*$", pd.NA, regex=True
)

print("Missing VIN:", df_clean["vin"].isna().sum())

Missing VIN: 199


In [39]:
df_clean.isna().sum().sort_values(ascending=False)

vin                      199
reporting_delay_clean     18
query_model                0
invalid_incident_date      0
reporting_delay            0
incident_date              0
filed_date                 0
model_year                 0
model                      0
make                       0
product_model              0
vehicle_match              0
query_model_year           0
odiNumber                  0
manufacturer               0
products                   0
summary                    0
components                 0
dateComplaintFiled         0
dateOfIncident             0
numberOfDeaths             0
numberOfInjuries           0
fire                       0
crash                      0
query_make                 0
dtype: int64

In [40]:
def get_vehicle_year(products):
    for p in products:
        if p.get("type") == "Vehicle":
            return p.get("productYear")
    return None

df_clean["product_year"] = df_clean["products"].apply(
    get_vehicle_year
)

df_clean["product_year"] = pd.to_numeric(
    df_clean["product_year"], errors="coerce"
)

In [41]:
df_clean["year_match"] = (
    df_clean["model_year"] == df_clean["product_year"]
)

print(df_clean["year_match"].value_counts())

display(
    df_clean.loc[
        ~df_clean["year_match"],
        ["odiNumber", "make", "model",
         "model_year", "product_year"]
    ].head(10)
)

year_match
True    9440
Name: count, dtype: int64


,odiNumber,make,model,model_year,product_year


In [42]:
display(
    df_clean.groupby(["make", "model", "model_year"])
    .size()
    .reset_index(name="complaint_count")
)

,make,model,model_year,complaint_count
0,AUDI,Q5,2018,207
1,AUDI,Q5,2019,52
2,AUDI,Q5,2020,25
3,BMW,3 SERIES,2018,59
4,BMW,3 SERIES,2019,41
5,BMW,3 SERIES,2020,53
6,FORD,ESCAPE,2018,1806
7,FORD,ESCAPE,2019,594
8,FORD,ESCAPE,2020,1533
9,FORD,F-150,2018,83


(maybe add a visualization here)

## Component Cleaning

In [43]:
df_clean["components"] = (
    df_clean["components"]
    .str.upper()
    .str.strip()
)

display(df_clean["components"].value_counts().head(20))

components
ENGINE                                1716
POWER TRAIN                            816
UNKNOWN OR OTHER                       761
STRUCTURE                              641
STEERING                               424
ENGINE AND ENGINE COOLING              371
ELECTRICAL SYSTEM                      350
FUEL SYSTEM, GASOLINE                  296
FUEL/PROPULSION SYSTEM                 269
SERVICE BRAKES                         207
AIR BAGS                               196
POWER TRAIN,ENGINE                     186
EXTERIOR LIGHTING                      135
FORWARD COLLISION AVOIDANCE            109
VISIBILITY/WIPER                       104
ENGINE,FUEL/PROPULSION SYSTEM           94
UNKNOWN OR OTHER,ENGINE                 91
ELECTRICAL SYSTEM,UNKNOWN OR OTHER      86
WHEELS                                  79
ELECTRICAL SYSTEM,ENGINE                74
Name: count, dtype: int64

In [44]:
print("Missing:", df_clean["components"].isna().sum())
print("Blank:", (df_clean["components"] == "").sum())

print("UNKNOWN OR OTHER:",
      (df_clean["components"] == "UNKNOWN OR OTHER").sum())

Missing: 0
Blank: 0
UNKNOWN OR OTHER: 761


In [47]:
print("Total rows:", len(df_clean))
print("Unique IDs:", df_clean["odiNumber"].nunique())
print("Duplicate IDs:", df_clean["odiNumber"].duplicated().sum())

print("Missing filing dates:", df_clean["filed_date"].isna().sum())

print("Missing vehicle keys:")
print(df_clean[["make", "model", "model_year"]].isna().sum())

print("Invalid incident dates:",
      df_clean["invalid_incident_date"].sum())

Total rows: 9440
Unique IDs: 9440
Duplicate IDs: 0
Missing filing dates: 0
Missing vehicle keys:
make          0
model         0
model_year    0
dtype: int64
Invalid incident dates: 18


## Export Cleaned Dataset

In [48]:
complaints_clean = df_clean.rename(columns={
    "odiNumber": "complaint_id",
    "numberOfInjuries": "number_of_injuries",
    "numberOfDeaths": "number_of_deaths"
}).copy()

# Use validated reporting delay for EDA
complaints_clean["reporting_delay"] = (
    complaints_clean["reporting_delay_clean"]
)

columns = [
    "complaint_id",
    "make",
    "model",
    "model_year",
    "product_model",
    "filed_date",
    "incident_date",
    "components",
    "crash",
    "fire",
    "number_of_injuries",
    "number_of_deaths",
    "summary",
    "reporting_delay",
    "invalid_incident_date"
]

complaints_clean = complaints_clean[columns]

display(complaints_clean.head())
print("Shape:", complaints_clean.shape)

,complaint_id,make,model,model_year,product_model,filed_date,incident_date,components,crash,fire,number_of_injuries,number_of_deaths,summary,reporting_delay,invalid_incident_date
0,11769969,TOYOTA,CAMRY,2018,CAMRY,2026-10-08,2026-10-06,ENGINE,False,False,0,0,My 2018 Toyota Camry LE went into Limp mode wh...,2.0,False
1,11769124,TOYOTA,CAMRY,2018,CAMRY,2026-10-05,2026-10-03,POWER TRAIN,False,False,0,0,The contact owns a 2018 Toyota Camry. The cont...,2.0,False
2,11768868,TOYOTA,CAMRY,2018,CAMRY,2026-10-04,2026-10-02,FUEL/PROPULSION SYSTEM,False,False,0,0,I am filing a formal complaint regarding a sev...,2.0,False
3,11768335,TOYOTA,CAMRY,2018,CAMRY,2026-10-01,2026-09-29,POWER TRAIN,False,False,0,0,"2018 Toyota Camry approx 126,000 miles. Trans...",2.0,False
4,11767971,TOYOTA,CAMRY,2018,CAMRY,2026-09-30,2026-09-28,STEERING,False,False,0,0,TSB-0028-22 - CAR DIAGNOSED AT DEALERSHIP WITH...,2.0,False


Shape: (9440, 15)


In [49]:
print("Rows:", len(complaints_clean))
print("Duplicates:", complaints_clean["complaint_id"].duplicated().sum())
print("Missing filed dates:", complaints_clean["filed_date"].isna().sum())
print("Invalid incident dates:", complaints_clean["invalid_incident_date"].sum())

assert len(complaints_clean) == len(df_raw)
assert complaints_clean["complaint_id"].is_unique
assert complaints_clean["filed_date"].notna().all()
assert df_clean["year_match"].all()

print("Validation passed!")

Rows: 9440
Duplicates: 0
Missing filed dates: 0
Invalid incident dates: 18
Validation passed!


In [50]:
output_file = PROCESSED_DIR / "complaints_clean.csv"

complaints_clean.to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: /Users/xinyishen/Desktop/Brown/DATA 1030/DATA1030-fall2026/DATA1030-Midterm-Xinfinity/data/processed/complaints_clean.csv


In [51]:
df_check = pd.read_csv(
    output_file,
    parse_dates=["filed_date", "incident_date"]
)

display(df_check.head())
print(df_check.shape)

,complaint_id,make,model,model_year,product_model,filed_date,incident_date,components,crash,fire,number_of_injuries,number_of_deaths,summary,reporting_delay,invalid_incident_date
0,11769969,TOYOTA,CAMRY,2018,CAMRY,2026-10-08,2026-10-06,ENGINE,False,False,0,0,My 2018 Toyota Camry LE went into Limp mode wh...,2.0,False
1,11769124,TOYOTA,CAMRY,2018,CAMRY,2026-10-05,2026-10-03,POWER TRAIN,False,False,0,0,The contact owns a 2018 Toyota Camry. The cont...,2.0,False
2,11768868,TOYOTA,CAMRY,2018,CAMRY,2026-10-04,2026-10-02,FUEL/PROPULSION SYSTEM,False,False,0,0,I am filing a formal complaint regarding a sev...,2.0,False
3,11768335,TOYOTA,CAMRY,2018,CAMRY,2026-10-01,2026-09-29,POWER TRAIN,False,False,0,0,"2018 Toyota Camry approx 126,000 miles. Trans...",2.0,False
4,11767971,TOYOTA,CAMRY,2018,CAMRY,2026-09-30,2026-09-28,STEERING,False,False,0,0,TSB-0028-22 - CAR DIAGNOSED AT DEALERSHIP WITH...,2.0,False


(9440, 15)


In [52]:
expected_model = df_clean["query_model"].str.upper().str.strip()

print("Model mismatches:",
      (df_clean["model"] != expected_model).sum())

display(
    df_clean.loc[
        df_clean["model"] != expected_model,
        ["odiNumber", "query_model", "product_model", "model"]
    ].head(10)
)

Model mismatches: 0


,odiNumber,query_model,product_model,model
